In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, QuantileTransformer

# load and clean data
df = pd.read_csv('dataset.csv')

df = df[df['duration_ms'] > 0]
df = df[df['tempo'] > 0]
df = df[df['time_signature'] > 0]
df = df.dropna(subset=['artists', 'album_name', 'track_name'])
df = df.drop_duplicates(subset=['track_name', 'artists'])
df = df.reset_index(drop=True)

# feature processing
qt = QuantileTransformer(output_distribution='uniform', random_state=42)
df['energy_scaled'] = qt.fit_transform(df[['energy']])

COST_FEATURES = ['valence', 'energy_scaled', 'danceability',
                 'acousticness', 'instrumentalness', 'tempo', 'speechiness']

scaler = MinMaxScaler()
df[COST_FEATURES] = scaler.fit_transform(df[COST_FEATURES])

CORE = ['valence', 'energy_scaled']

features_core = df[CORE].values
features_cost = df[COST_FEATURES].values

# penalty weights
WEIGHTS = np.array([
    1.0,   # valence
    1.0,   # energy_scaled
    0.4,   # danceability
    0.4,   # acousticness
    0.3,   # instrumentalness
    0.2,   # tempo
    0.1,   # speechiness
])

# distance to target
def dist_to_target(song_idx, target):
    diff = features_core[song_idx] - target
    return np.sqrt(np.sum(diff ** 2))

# hueristic
def h(song_idx, target, current_dist, ideal_step):
    ideal_next_dist = max(current_dist - ideal_step, 0)
    actual_dist     = dist_to_target(song_idx, target)
    return abs(actual_dist - ideal_next_dist)

# transition cost
def transition_cost(idx_a, idx_b):
    diff = features_cost[idx_a] - features_cost[idx_b]
    return np.sqrt(np.sum(WEIGHTS * diff ** 2))

# direction penalty
def direction_penalty(song_idx, current_idx, target):
    current_v, current_e = features_core[current_idx]
    target_v,  target_e  = target
    cand_v,    cand_e    = features_core[song_idx]

    # Positive = moving away from target on that axis
    v_dev = abs(cand_v - target_v) - abs(current_v - target_v)
    e_dev = abs(cand_e - target_e) - abs(current_e - target_e)

    # Only penalize wrong-direction moves, not reward correct ones
    return max(v_dev, 0) * 1.5 + max(e_dev, 0) * 1.5

# candidate pool
def get_candidates(song_idx, target, current_dist, ideal_step, n=30):
    dists           = np.linalg.norm(features_core - target, axis=1)
    ideal_next_dist = max(current_dist - ideal_step, 0)

    closer_idxs = np.where(dists < current_dist)[0]

    if len(closer_idxs) == 0:
        d = np.linalg.norm(features_core - features_core[song_idx], axis=1)
        d[song_idx] = np.inf
        return np.argsort(d)[:n]

    deviation = np.abs(dists[closer_idxs] - ideal_next_dist)
    top_n     = np.argsort(deviation)[:n]
    return closer_idxs[top_n]

# greedy best first implementation
def greedy_playlist(start_idx, target, max_songs=10, n_candidates=30, threshold=0.08):
    total_dist = dist_to_target(start_idx, target)
    ideal_step = total_dist / max_songs
    print(f"Total distance: {total_dist:.3f} | Ideal step size: {ideal_step:.3f}")

    path    = [start_idx]
    visited = {start_idx}
    current = start_idx

    while len(path) < max_songs:
        current_dist = dist_to_target(current, target)

        # terminate if close enough to target
        if current_dist < threshold:
            break

        candidates = get_candidates(current, target, current_dist,
                                    ideal_step, n=n_candidates)
        candidates = [c for c in candidates if c not in visited]

        if not candidates:
            break

        best      = None
        best_score = np.inf

        for c in candidates:
            score = (h(c, target, current_dist, ideal_step)
                   + transition_cost(current, c)
                   + direction_penalty(c, current, target))
            if score < best_score:
                best_score = score
                best       = c

        path.append(best)
        visited.add(best)
        current = best

    return path

# mood label --> coordinate mapping
MOOD_MAP = {
    'happy':     np.array([0.85, 0.75]),
    'sad':       np.array([0.15, 0.20]),
    'energetic': np.array([0.60, 0.95]),
    'calm':      np.array([0.55, 0.15]),
    'angry':     np.array([0.10, 0.90]),
    'relaxed':   np.array([0.75, 0.30]),
}

def mood_to_coord(mood_str):
    mood_str = mood_str.lower().strip()
    if mood_str not in MOOD_MAP:
        print(f"Unknown mood '{mood_str}'. Available: {list(MOOD_MAP.keys())}")
        return None
    return MOOD_MAP[mood_str]

# display results
def show_playlist(path, target):
    print(f"\n🎵 Generated Playlist ({len(path)} songs)\n")
    print(f"{'#':<4} {'Track':<35} {'Artist':<25} {'valence':>8} {'energy':>8} {'to target':>10}")
    print("─" * 97)
    for i, idx in enumerate(path):
        row = df.iloc[idx]
        d   = dist_to_target(idx, target)
        print(f"{i+1:<4} {str(row['track_name']):<35} "
              f"{str(row['artists']):<25} "
              f"{row['valence']:>8.3f} {row['energy']:>8.3f} {d:>10.3f}")

# example
if __name__ == '__main__':

    start_idx  = 0
    start_song = df.iloc[start_idx]
    print(f"Starting song: {start_song['track_name']} — {start_song['artists']}")
    print(f"Starting mood: valence={start_song['valence']:.3f}, energy={start_song['energy']:.3f}")

    target_mood = 'energetic'
    target      = mood_to_coord(target_mood)
    print(f"\nTarget mood: {target_mood} → {target}")

    path = greedy_playlist(
        start_idx    = start_idx,
        target       = target,
        max_songs    = 10,
        n_candidates = 30,
        threshold    = 0.08
    )

    show_playlist(path, target)

Starting song: Comedy — Gen Hoshino
Starting mood: valence=0.719, energy=0.461

Target mood: energetic → [0.6  0.95]
Total distance: 0.704 | Ideal step size: 0.070

🎵 Generated Playlist (10 songs)

#    Track                               Artist                     valence   energy  to target
─────────────────────────────────────────────────────────────────────────────────────────────────
1    Comedy                              Gen Hoshino                  0.719    0.461      0.704
2    Ee Raathale (From "Radhe Shyam")    Yuvan Shankar Raja;Harini Ivaturi;Justin Prabhakaran    0.554    0.528      0.634
3    Relax (Your Mind)                   Blank & Jones;Jason Caesar    0.571    0.593      0.563
4    Empty Lightning                     Woesum;Oklou                 0.583    0.651      0.493
5    Alunguraen Kulunguraen              Prasanna;Namitha Babu        0.601    0.703      0.422
6    Be Your Friend‬‬                    Vigiland;Alexander Tidebrink    0.644    0.754      0.352
7